# Tập 2.4 · Hai mắt để đoán khoảng cách (stereo) 👀

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap24/tap24_stereo.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 2 · ⏱ ~40 phút · Giáo trình: Principles of Robot Autonomy, [Chương 7](https://porabook.com/book/stereo-vision/)

Hai camera cạnh nhau: vật càng gần càng lệch nhiều giữa hai ảnh. Trong notebook này bạn:

1. Chạy lại đúng mô phỏng trong video (dựng 2 ảnh và bản đồ độ sâu, mất vài giây).
2. **Tự viết** công thức độ sâu, so khớp trên một hàng và nội suy dưới pixel (3 bài), có ô chấm tự động.
3. Vẽ bản đồ độ sâu và đường chi phí bằng hàm của bạn.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap24")
sys.path[:0] = ["bai_tap", "sim", ".."]

import numpy as np
import stereo as goc               # mô phỏng trong video
from ve_do_thi import ve           # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
print(f"Camera {goc.W}×{goc.H}, f = {goc.F:.0f} px, baseline b = {goc.B * 100:.0f} cm")
for t in goc.THONG_KE:
    print(f"  {t['ten']:9s} z thật {t['z_that']:.2f} m · d thật {t['d_that']:.2f} px · d đo {t['d_do'] or 0:.2f} px · z đo {t['z_do'] or 0:.3f} m · bị loại {t['loai_bo'] * 100:.0f}%")

## 2. Bài tập

Độ sâu từ disparity: $z = b\,f/d$. So khớp: với mỗi $d$, trung bình bình phương hiệu giữa cửa sổ quanh $(hàng, cột)$ ở ảnh trái và quanh $(hàng, cột - d)$ ở ảnh phải. Nội suy: $\delta = \tfrac12 (c_0 - c_2)/(c_0 - 2c_1 + c_2)$.

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `do_sau(d, b, f)` | $z = b f / d$ (số hoặc mảng) |
| **2** | `chi_phi_diem(L, R, hang, cot, d_max, cua_so)` | mảng `d_max` chi phí |
| **3** | `noi_suy_parabol(c0, c1, c2)` | độ lệch dưới pixel, kẹp trong ±0,5 |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import numpy as np


def do_sau(d, b, f):
    """Bài 1 (Eq. 7.7): điểm lệch d pixel giữa hai ảnh, hai camera cách nhau b mét, tiêu cự f pixel → độ sâu z (mét).

        z = b·f / d       (d có thể là số hoặc mảng numpy)"""
    raise NotImplementedError("TODO Bài 1: z = b·f/d")


def chi_phi_diem(L, R, hang, cot, d_max, cua_so):
    """Bài 2: so khớp một pixel (hang, cot) của ảnh trái với các pixel cùng hàng của ảnh phải.

    Với mỗi d = 0, 1, ..., d_max − 1: lấy cửa sổ cua_so × cua_so quanh (hang, cot) trong L và quanh (hang, cot − d) trong R,
    tính TRUNG BÌNH bình phương hiệu (mean squared difference) của hai cửa sổ.
    Gợi ý: r = cua_so // 2; cửa sổ trái = L[hang−r : hang+r+1, cot−r : cot+r+1].
    Trả về mảng numpy độ dài d_max; d có chi phí nhỏ nhất là độ lệch (disparity) tìm được."""
    raise NotImplementedError("TODO Bài 2: chi phí theo d")


def noi_suy_parabol(c0, c1, c2):
    """Bài 3 (mở rộng): chi phí tại d−1, d, d+1 là c0, c1, c2 (c1 nhỏ nhất). Vẽ parabol qua 3 điểm, đỉnh parabol lệch khỏi d một đoạn

        δ = 0.5·(c0 − c2) / (c0 − 2·c1 + c2)
    Kẹp δ trong [−0.5, 0.5]; nếu mẫu số ≤ 0 thì trả về 0. Trả về δ (pixel) để có d + δ chính xác dưới 1 pixel."""
    raise NotImplementedError("TODO Bài 3: độ lệch dưới pixel")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
import loi_giai
try:
    ve(globals(), "ket_qua.png")
except NotImplementedError:
    print("Chưa xong Bài 1–2, vẽ bằng lời giải.")
    ve(loi_giai, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Đổi kích thước cửa sổ so khớp: đáy của đường chi phí rõ hay mờ đi?

In [ ]:
import loi_giai
for cs in [3, 7, 11, 21, 31]:
    c = loi_giai.chi_phi_diem(goc.L, goc.R, goc.DIEM[0], goc.DIEM[1], goc.D_MAX, cs)
    d0 = int(np.argmin(c))
    nhi = np.partition(c[np.abs(np.arange(goc.D_MAX) - d0) > 3], 0)[0]
    print(f"cửa sổ {cs:2d}×{cs:<2d}: d = {d0} px, chi phí đáy {c[d0]:7.1f}, đáy thứ hai (cách ≥ 4 px) {nhi:7.1f}")

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap24).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)